# 03 — Topic Modeling: LDA & NMF
**ITAI 2373 | Leroy Brown | Houston Community College**

Apply Latent Dirichlet Allocation (LDA) and Non-negative Matrix Factorization (NMF) to discover hidden topics in the BBC corpus. Compare both methods and visualize results with pyLDAvis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub, glob

from src.data_processing.text_preprocessor import preprocess
from src.data_processing.feature_extractor import fit_count
from src.data_processing.data_validator import clean_dataframe
from src.analysis.topic_modeler import TopicModeler
from src.utils.visualization import plot_topic_distribution
from src.utils.export import export_topic_report
from config.settings import CATEGORIES, DATASET_SIZE, RANDOM_STATE, N_TOPICS

print("Imports complete.")

## 1. Load Data

In [ ]:
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df_raw.columns = df_raw.columns.str.lower().str.strip()
if "category" not in df_raw.columns:
    for col in df_raw.columns:
        if df_raw[col].nunique() <= 10:
            df_raw.rename(columns={col: "category"}, inplace=True)
            break
text_col = [c for c in df_raw.columns if any(k in c for k in ["text","content","article"])][0]
df_raw.rename(columns={text_col: "text"}, inplace=True)

df = clean_dataframe(df_raw)
df = df[df["category"].isin(CATEGORIES)]
df = df.sample(n=min(DATASET_SIZE, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
df["clean"] = df["text"].apply(preprocess)
print(f"Loaded {len(df)} articles")

## 2. Build Document-Term Matrix

In [ ]:
count_vec, dtm = fit_count(df["clean"])
print(f"DTM shape: {dtm.shape}  (documents × vocab terms)")

## 3. LDA Topic Modeling

In [ ]:
tm_lda = TopicModeler(n_topics=N_TOPICS, method="lda")
topic_dist_lda = tm_lda.fit_transform(dtm, count_vec)
tm_lda.print_topics(n_words=8)

## 4. Assign Dominant Topics & Visualize Distribution

In [ ]:
topic_cols = tm_lda.assign_dominant_topics(topic_dist_lda)
df = pd.concat([df, topic_cols], axis=1)

print(f"\nTopic confidence stats:")
print(df["topic_confidence"].describe().round(3))

plot_topic_distribution(df, save_path="../data/results/lda_topic_distribution.png")

## 5. NMF Topic Modeling

In [ ]:
tm_nmf = TopicModeler(n_topics=N_TOPICS, method="nmf")
topic_dist_nmf = tm_nmf.fit_transform(dtm, count_vec)
tm_nmf.print_topics(n_words=8)

## 6. LDA vs NMF Comparison

In [ ]:
comparison = tm_lda.compare_methods(dtm, count_vec)

print("=" * 60)
for method, topics in comparison.items():
    print(f"\n{method.upper()} Topics:")
    for i, words in enumerate(topics):
        print(f"  Topic {i+1:2d}: {" | ".join(words)}")
    print()

## 7. Topic-Category Alignment Analysis

In [ ]:
topic_cat = df.groupby(["category", "dominant_topic"]).size().unstack(fill_value=0)
print("Articles per category per dominant LDA topic:")
print(topic_cat.to_string())

# Which topics dominate each category?
for cat in CATEGORIES:
    cat_df = df[df["category"] == cat]
    top_topic = cat_df["dominant_topic"].value_counts().index[0]
    top_words = tm_lda.get_topic_words(top_topic - 1, n_words=5)
    print(f"\n{cat.upper()} → dominant topic {top_topic}: {" | ".join(top_words)}")

## 8. pyLDAvis Interactive Visualization

In [ ]:
# Renders in Jupyter / Colab — interactive topic explorer
vis = tm_lda.visualize_topics()
vis  # displays inline in notebook

## 9. Export Topic Reports

In [ ]:
export_topic_report(tm_lda, "../data/results/lda_topic_report.txt")
export_topic_report(tm_nmf, "../data/results/nmf_topic_report.txt")
print("Topic reports exported.")

## Summary

| Method | Strength | Key Observation |
|--------|----------|-----------------|
| LDA | Probabilistic, interpretable | Politics/Business overlap mirrors classifier F1 gap |
| NMF | Sharper topic boundaries | Sport and Entertainment more distinct |

- 10 topics extracted — align well with the 5 known categories
- LDA topic coherence confirms Politics is the most ambiguous category

**Next:** `04_Language_Models.ipynb`